# Scientific validation using local run for synthetic small dataset 

Pipeline on the Zenodo synthetic example (37 buildings)


## 1. Setup and paths


In [ ]:
import os, sys, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO))
from urban_pfr.analyzer import PFRAnalyzer

# Paths, which DATA_DIR must contain the methodology-paper example data (Zenodo record):
#   PluvialFloodRiskMap_Data.lpkx   
#   pluvialfloodriskmap.gdb        
DATA_DIR = Path(os.environ.get('PFR_DATA_DIR', REPO / 'data'))
LPKX = Path(os.environ.get('PFR_LPKX', DATA_DIR / 'PluvialFloodRiskMap_Data.lpkx'))
FALLBACK_GDB = Path(os.environ.get('PFR_GDB', DATA_DIR / 'pluvialfloodriskmap.gdb'))
BOUNDARY_DIR = Path(os.environ.get('PFR_BOUNDARY_DIR', REPO / 'output' / 'boundary_cache'))

OUT = REPO / 'output' / 'scientific_validation_synthetic_small'
RUN_DIR = OUT / 'run'
OUT.mkdir(parents=True, exist_ok=True)
BOUNDARY_DIR.mkdir(parents=True, exist_ok=True)

GDB = FALLBACK_GDB
if LPKX.exists():
    try:
        import py7zr
        EXTRACT = OUT / 'lpkx_extracted'
        if EXTRACT.exists():
            shutil.rmtree(EXTRACT)
        with py7zr.SevenZipFile(LPKX, 'r') as z:
            z.extractall(EXTRACT)
        GDB = sorted(EXTRACT.rglob('*.gdb'))[0]
    except ImportError:
        pass
assert Path(GDB).exists(), f'example data not found: set PFR_LPKX or PFR_GDB (tried {GDB})'
print('GDB :', GDB)
print('OUT :', OUT)


## 2. Full pipeline 


This validation depends on a version-aligned set of reference code, configuration, input data, and outputs.

### 2.1 Config

In [ ]:
STAGED = OUT / 'inputs'
FLOODS = STAGED / 'Floodlevels'
FLOODS.mkdir(parents=True, exist_ok=True)

b = gpd.read_file(GDB, layer='Building_ExampleLayer')
b = b.drop(columns=[c for c in ['Sen', 'R', 'R_G'] if c in b.columns])
b.to_file(STAGED / 'buildings.gpkg', driver='GPKG')
gpd.read_file(GDB, layer='StatisticalExampleUnit').to_file(STAGED / 'statistical_units.gpkg', driver='GPKG')
gpd.read_file(GDB, layer='Streets').to_file(STAGED / 'streets.gpkg', driver='GPKG')
for d in range(30, 101, 10):
    gpd.read_file(GDB, layer=f'Flood_{d}').to_file(FLOODS / f'Flood_{d}.gpkg', driver='GPKG')

config = {
  'project': {'city_name': 'scientific_validation_synthetic_small', 'crs': 'EPSG:3857',
    'paths': {'input_gdb': None,
              'buildings': str(STAGED / 'buildings.gpkg'),
              'statistical_units': str(STAGED / 'statistical_units.gpkg'),
              'streets': str(STAGED / 'streets.gpkg'),
              'flood_dir': str(FLOODS),
              'output_dir': str(RUN_DIR)}},
  'topsis_granularity': 'building',
  'schema': {'stat_unit_col': 'StatisticalUnit', 'residents_col': 'Residents',
             'living_area_col': 'LivingArea', 'floors_col': 'Floors',
             'building_type_col': 'Building_type',
             'sensitivity_fields': ['ES', 'C'], 'coping_fields': ['WR', 'EDQ']},
  'hazard_settings': {'hma_buffers': [5, 15, 30], 'hwb_buffer': 2, 'flood_threshold': 0.3,
                      'flood_depths': [30, 40, 50, 60, 70, 80, 90, 100],
                      'shape_param': 0.25, 'min_area_threshold': 0.0, 'hwb_clip_max': None},
  'risk_settings': {'weight_hazard': 1.0, 'weight_exposure': 1.0, 'weight_vulnerability': 1.0,
                    'normalize_exposure': False},
  'topsis_weights': {'sensitivity': [0.7, 0.3], 'coping_capacity': [0.5, 0.5], 'svi': [0.5, 0.5]},
  'svpf_threshold': 0.25, 'svpf_transform': 2.0,
}

analyzer = PFRAnalyzer(config)
analyzer.load_data()
analyzer.run_pipeline()
print('pipeline finished')

## 3. Numeric validation vs the ArcGIS golden layer


In [ ]:
FIELDS = ['Sen', 'CCap', 'SV', 'SV_PF', 'EMA', 'EWB', 'HMA', 'HWB', 'PFR_WB', 'PFR_MA']
ref = gpd.read_file(GDB, layer='Building_ExampleLayer_Results').drop(columns='geometry')
ref['ID'] = ref['ID'].astype(int)

out = analyzer.buildings
result = pd.DataFrame({'ID': out['ID'].astype(int),
    'Sen': out['Sensitivity'], 'CCap': out['CopingCapacity'], 'SV': out['SVI'], 'SV_PF': out['SVPF'],
    'EMA': out['R'], 'EWB': out['R_G'], 'HMA': out['HMA'], 'HWB': out['HWB'],
    'PFR_WB': out['PFRWB'], 'PFR_MA': out['PFRMA']})

m = ref.merge(result, on='ID', suffixes=('_ref', '_py'))
rows = []
for f in FIELDS:
    r, p = m[f + '_ref'], m[f + '_py']
    d = (p - r).abs()
    rows.append(dict(field=f, ref_mean=r.mean(), py_mean=p.mean(),
                     ref_median=r.median(), py_median=p.median(),
                     MAE=d.mean(), max_abs_err=d.max(), correlation=r.corr(p),
                     verdict='exact' if d.max() <= 1e-6 else ('near' if r.corr(p) >= 0.95 else 'CHECK')))
summary = pd.DataFrame(rows)
summary.to_csv(OUT / 'comparison_summary.csv', index=False)
print(f'{len(m)} buildings compared | summary -> {OUT / "comparison_summary.csv"}')
print(summary.round(4).to_string(index=False))

In [ ]:
pfrma = out['SVPF'].fillna(0) * out['R'].fillna(0) * out['HMA'].fillna(0)
pfrwb = out['SVPF'].fillna(0) * out['R_G'].fillna(0) * out['HWB'].fillna(0)
print('PFRMA = SVPF x R   x HMA : max abs diff %.3g' % (pfrma - out['PFRMA']).abs().max())
print('PFRWB = SVPF x R_G x HWB : max abs diff %.3g' % (pfrwb - out['PFRWB']).abs().max())

paths = analyzer.save_results()
saved = gpd.read_file(paths['buildings_private'])
footprints = gpd.read_file(STAGED / 'buildings.gpkg')
assert saved.geometry.geom_equals(footprints.geometry).all()
print('OK: private output stores the original building footprints (geometry-lifecycle fix)')
for k, v in paths.items():
    print(' ', k, '->', Path(v).relative_to(OUT) if str(v).startswith(str(OUT)) else v)

In [ ]:
## TODO add visualization and comparison